# Предобученная модель YOLO для детектинга людей  



### Зависимости  

In [2]:
import os
import cv2
from ultralytics import YOLO

#plots
import matplotlib.pyplot as plt
import seaborn as sns

#basics
import pandas as pd
import numpy as np
import subprocess

import time
# from tqdm.notebook import tqdm

# Display image and videos
import IPython
from IPython.display import Video, display
%matplotlib inline


import urllib.request 
import shutil

In [3]:
# Загрузка предобученной модели YOLOv11
model = YOLO("runs/result/best.pt")

In [2]:
# Загрузка изображения
#image = cv2.imread('2.png')
image = cv2.imread('C:/Users/Egor/Desktop/ss/AI-Flow-Detecting/ai-core/dataset/test/images/20250810_130413.jpg')

# Детекция людей (класс '0' в COCO = человек)
results = model(image, classes=[0])  

# Визуализация результатов
annotated_image = results[0].plot()  # Рисует bounding boxes
cv2.imshow('Detected People', annotated_image)
cv2.waitKey(0)
cv2.destroyAllWindows()

# Подсчёт количества людей
people_count = len(results[0].boxes)
print(f"Количество людей на фото: {people_count}")



0: 384x640 17 humans, 44.4ms
Speed: 2.4ms preprocess, 44.4ms inference, 23.8ms postprocess per image at shape (1, 3, 384, 640)
Количество людей на фото: 17


In [3]:
from ultralytics import YOLO
import cv2

# Загрузка модели YOLOv8 (nano — быстро, подходит для реального времени)
#model = YOLO('yolov10n.pt')

# URL видеопотока (убраны лишние пробелы в конце!)
# https://restreamer.vms.evo73.ru/918335436b92ac26/stream.m3u8
# https://restreamer.vms.evo73.ru/24c3036fe19a150a/stream.m3u8
stream_url = "https://restreamer.vms.evo73.ru/918335436b92ac26/stream.m3u8"

# Открываем видеопоток
cap = cv2.VideoCapture(stream_url)

if not cap.isOpened():
    print("❌ Ошибка: Не удалось открыть видеопоток. Проверь URL и соединение.")
    exit()

print("✅ Видеопоток запущен. Детекция людей через YOLOv8...")

while True:
    ret, frame = cap.read()

    if not ret:
        print("⚠️ Не удалось получить кадр. Поток может быть разорван.")
        break

    # Детекция ТОЛЬКО людей (класс 0 в COCO)
    results = model.predict(frame, classes=[0], conf=0.5, verbose=False)

    # Наносим bounding boxes и метки
    annotated_frame = results[0].plot()

    # Подсчёт обнаруженных людей
    people_count = len(results[0].boxes)
    cv2.putText(annotated_frame, f'Людей: {people_count}', (10, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 2, cv2.LINE_AA)

    # Показываем кадр
    cv2.imshow('YOLOv8 — Детекция людей в реальном времени', annotated_frame)

    # Выход по клавише 'q' или пробелу
    if cv2.waitKey(1) & 0xFF in [ord('q'), ord(' ')]:
        break

# Освобождение ресурсов
cap.release()
cv2.destroyAllWindows()
print("⏹️ Работа завершена.")

✅ Видеопоток запущен. Детекция людей через YOLOv8...
⏹️ Работа завершена.


# Парсинг  

In [ ]:
os.makedirs("data/images", exist_ok=True)

cap = cv2.VideoCapture("https://restreamer.vms.evo73.ru/918335436b92ac26/stream.m3u8")
count = 0

print("Сбор кадров начат... Нажми 'q' для остановки.")

while True:
    ret, frame = cap.read()
    if ret:
        if count % 60 == 0:  # один кадр каждые 2 сек (30 FPS)
            cv2.imwrite(f"data/images/frame_{count}.jpg", frame)
            print(f"Сохранён кадр {count}")
        count += 1

    if cv2.waitKey(1) & 0xFF == ord('q') or count > 500:  # 500 кадров
        break

cap.release()
cv2.destroyAllWindows()

Сбор кадров начат... Нажми 'q' для остановки.
Сохранён кадр 0
Сохранён кадр 60
Сохранён кадр 120
Сохранён кадр 180
Сохранён кадр 240
Сохранён кадр 300
Сохранён кадр 360
Сохранён кадр 420
Сохранён кадр 480


In [3]:
# detect_on_stream.py
from ultralytics import YOLO
import cv2
import numpy as np
import pandas as pd
import os
import subprocess

# -------------------------------
# ### Конфигурации
# -------------------------------
MODEL_PATH = 'runs/result/preprocessdetect2.pt'  # твоя дообученная модель
STREAM_URL = "https://restreamer.vms.evo73.ru/918335436b92ac26/stream.m3u8"
CONF_LEVEL = 0.6
SCALE_PERCENT = 100
VIDEO_NAME = "result.mp4"
ROI_POINTS = np.array([(1250, 400), (750, 400), (700, 800), (1200, 800)], np.int32)
ALPHA = 0.1
PATIENCE = 100
FRAME_MAX = 5
THR_CENTERS = 25

# -------------------------------
# Загрузка модели
# -------------------------------
model = YOLO(MODEL_PATH)
dict_classes = model.model.names

# -------------------------------
# Открываем поток
# -------------------------------
cap = cv2.VideoCapture(STREAM_URL)
if not cap.isOpened():
    print("❌ Не удалось открыть поток")
    exit()

# Параметры видео
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
fps = cap.get(cv2.CAP_PROP_FPS)

if SCALE_PERCENT != 100:
    width = int(width * SCALE_PERCENT / 100)
    height = int(height * SCALE_PERCENT / 100)

# -------------------------------
# VideoWriter
# -------------------------------
tmp_out = "tmp_result.mp4"
output_out = "rep_result.mp4"

if os.path.exists(tmp_out): os.remove(tmp_out)
if os.path.exists(output_out): os.remove(output_out)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(tmp_out, fourcc, fps, (width, height))

# -------------------------------
# Вспомогательные функции
# -------------------------------
def resize_frame(frame, scale):
    if scale == 100: return frame
    w = int(frame.shape[1] * scale / 100)
    h = int(frame.shape[0] * scale / 100)
    return cv2.resize(frame, (w, h), interpolation=cv2.INTER_AREA)

def update_tracking(centers_old, center, thr, last_key, frame_idx, frame_max):
    best_id = None
    min_dist = float('inf')
    for obj_id, history in centers_old.items():
        last_center = list(history.keys())[-1]
        dist = np.linalg.norm(np.array(last_center) - np.array(center))
        if dist < thr and dist < min_dist:
            min_dist = dist
            best_id = obj_id

    if best_id is not None:
        centers_old[best_id][center] = frame_idx
        return centers_old, best_id, False, best_id
    else:
        new_id = f"p_{len(centers_old)}"
        centers_old[new_id] = {center: frame_idx}
        return centers_old, new_id, True, new_id

def filter_tracks(centers_old, patience):
    current_frame = max([max(t.values()) for t in centers_old.values()] + [0])
    to_remove = [k for k, v in centers_old.items() if current_frame - max(v.values()) > patience]
    for k in to_remove:
        del centers_old[k]
    return centers_old

# -------------------------------
# Основной цикл
# -------------------------------
centers_old = {}
total_count = 0
frame_idx = 0

print("✅ Обработка начата. Нажмите 'q' для остановки.")

while True:
    ret, frame = cap.read()
    if not ret:
        print("⚠️ Кадр не получен. Переподключение...")
        cap.release()
        cap = cv2.VideoCapture(STREAM_URL)
        continue

    frame_idx += 1
    frame = resize_frame(frame, SCALE_PERCENT)
    display_frame = frame.copy()

    # --- ROI ---
    overlay = display_frame.copy()
    cv2.polylines(overlay, [ROI_POINTS], True, (255, 0, 0), 2)
    cv2.fillPoly(overlay, [ROI_POINTS], (255, 0, 0))
    cv2.addWeighted(overlay, ALPHA, display_frame, 1 - ALPHA, 0, display_frame)

    # Вырезаем ROI
    x_min, y_min = ROI_POINTS[:, 0].min(), ROI_POINTS[:, 1].min()
    x_max, y_max = ROI_POINTS[:, 0].max(), ROI_POINTS[:, 1].max()
    roi_frame = frame[y_min:y_max, x_min:x_max]

    # --- Детекция ---
    results = model(roi_frame, conf=CONF_LEVEL, classes=[0], verbose=False)
    if len(results[0].boxes) == 0:
        detections = pd.DataFrame(columns=['xmin', 'ymin', 'xmax', 'ymax', 'conf'])
    else:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        conf = results[0].boxes.conf.cpu().numpy()
        detections = pd.DataFrame(np.column_stack([boxes, conf]),
                                  columns=['xmin', 'ymin', 'xmax', 'ymax', 'conf'])

    # --- Обработка каждого человека ---
    for _, row in detections.iterrows():
        xmin, ymin, xmax, ymax, conf_val = map(int, row)
        cx = (xmin + xmax) // 2 + x_min
        cy = (ymin + ymax) // 2 + y_min

        centers_old, obj_id, is_new, _ = update_tracking(
            centers_old, (cx, cy), THR_CENTERS, None, frame_idx, FRAME_MAX
        )
        total_count += is_new

        # Рисуем
        cv2.rectangle(display_frame, (xmin + x_min, ymin + y_min), (xmax + x_min, ymax + y_min), (0, 255, 0), 2)
        cv2.circle(display_frame, (cx, cy), 5, (0, 255, 0), -1)
        cv2.putText(display_frame, f"{obj_id} ({conf_val:.2f})", (xmin + x_min, ymin + y_min - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 1)

    # Фильтрация
    centers_old = filter_tracks(centers_old, PATIENCE)

    # Счётчик
    cv2.putText(display_frame, f"Людей в зоне: {total_count}", (30, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 255), 2)

    # Запись и отображение
    out.write(display_frame)
    cv2.imshow("YOLOv8 — Пассажиропоток", display_frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# -------------------------------
# Финализация
# -------------------------------
out.release()
cap.release()
cv2.destroyAllWindows()

os.remove(tmp_out)
print(f"✅ Видео сохранено: {output_out}")

✅ Обработка начата. Нажмите 'q' для остановки.
✅ Видео сохранено: rep_result.mp4
